# 12 - Fusion Multimodale Texte-Image

## Objectif de ce notebook

Combiner les **features textuelles** et les **features visuelles** pour améliorer la classification de produits.

**Approches testées :**
1. **Late Fusion** : concaténation TF-IDF + features image (ResNet50, CLIP)
2. **CLIP Multimodal** : texte et image encodés dans le même espace par CLIP
3. **Démonstration de matching** : retrouver une image à partir d'un texte (et inversement)

**Comparaison** avec les résultats texte-seul (NB04–06) et image-seule (NB09–11).

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = str(os.cpu_count() or 4)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import pickle
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

sys.path.append(str(Path('../').resolve()))

from sklearn.preprocessing import LabelEncoder, normalize
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import train_test_split

from src.multimodal import (
    load_text_image_pairs, create_pairs_dataset,
    compute_similarity, find_matching_images, find_matching_texts, recall_at_k
)
from src.modeling import TFIDFVectorizer
from src.image import ImageFeatureExtractor
from src.evaluation import evaluate_model

import torch
import open_clip
from PIL import Image

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

plt.style.use('seaborn-v0_8')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

print(f'Device: {DEVICE}')

## 1. Chargement et alignement des données

Fusion des données textuelles et visuelles via un **merge** sur `(productid, imageid)`. Seuls les produits possédant à la fois un texte et une image nettoyée sont conservés.

In [ ]:
ROOT = Path('../').resolve()
DATA_BRUT = ROOT / 'data brut'
IMAGE_CLEAN_DIR = ROOT / 'data' / 'processed' / 'image_clean'
MODELS_DIR = ROOT / 'models'
MODELS_DIR.mkdir(exist_ok=True)

MULTIMODAL_CACHE = MODELS_DIR / 'multimodal_features_cache.npz'

print('Chargement des paires texte-image...')
pairs_df = load_text_image_pairs(DATA_BRUT, IMAGE_CLEAN_DIR, root=ROOT)
print(f'Paires texte-image : {len(pairs_df):,} produits')
print(f'Classes : {pairs_df["prdtypecode"].nunique()}')
print(f'\nDistribution des classes :')
print(pairs_df['prdtypecode'].value_counts().describe())

# Split train/val stratifi\u00e9
train_df, val_df = create_pairs_dataset(pairs_df, train_size=0.8, random_state=SEED)
print(f'\nTrain : {len(train_df):,} | Val : {len(val_df):,}')

# Label encoding
le = LabelEncoder()
y_train = le.fit_transform(train_df['prdtypecode'].values)
y_val = le.transform(val_df['prdtypecode'].values)
NUM_CLASSES = len(le.classes_)
print(f'Classes encod\u00e9es : {NUM_CLASSES}')

## 2. Extraction des features

Trois types de features extraits :
- **TF-IDF** (texte) : réduit à 300 dims via TruncatedSVD
- **ResNet50** (image) : 2048 dims, backbone figé ImageNet
- **CLIP** (image + texte) : 512 dims chacun, espace partagé

In [ ]:
# TF-IDF sur le texte combin\u00e9 (designation + description)
print('Extraction TF-IDF...')
tfidf = TFIDFVectorizer(max_features=10000, min_df=2, max_df=0.95, ngram_range=(1, 2))
X_tfidf_train_sparse = tfidf.fit_transform(pd.Series(train_df['text'].values))
X_tfidf_val_sparse = tfidf.transform(pd.Series(val_df['text'].values))
print(f'  TF-IDF brut : {X_tfidf_train_sparse.shape}')

# R\u00e9duction de dimension via TruncatedSVD
SVD_DIMS = 300
svd = TruncatedSVD(n_components=SVD_DIMS, random_state=SEED)
X_tfidf_train = svd.fit_transform(X_tfidf_train_sparse)
X_tfidf_val = svd.transform(X_tfidf_val_sparse)
explained = svd.explained_variance_ratio_.sum()
print(f'  TF-IDF r\u00e9duit : {X_tfidf_train.shape} (variance expliqu\u00e9e : {explained:.1%})')

# Normalisation L2
X_tfidf_train = normalize(X_tfidf_train)
X_tfidf_val = normalize(X_tfidf_val)

In [ ]:
RESNET_CACHE = MODELS_DIR / 'multimodal_resnet_cache.npz'

if RESNET_CACHE.exists():
    print('Chargement ResNet50 features depuis le cache...')
    cache = np.load(RESNET_CACHE)
    X_resnet_train = cache['X_train']
    X_resnet_val = cache['X_val']
else:
    print('Extraction ResNet50 features...')
    extractor = ImageFeatureExtractor(output_dim=2048)
    X_resnet_train = extractor.extract_from_paths(
        train_df['image_path'].tolist(), base_dir=ROOT, batch_size=32, show_progress=True
    )
    X_resnet_val = extractor.extract_from_paths(
        val_df['image_path'].tolist(), base_dir=ROOT, batch_size=32, show_progress=True
    )
    np.savez(RESNET_CACHE, X_train=X_resnet_train, X_val=X_resnet_val)
    print(f'Cache sauvegard\u00e9 : {RESNET_CACHE}')

X_resnet_train = normalize(X_resnet_train)
X_resnet_val = normalize(X_resnet_val)
print(f'ResNet50 features : train={X_resnet_train.shape}, val={X_resnet_val.shape}')

In [ ]:
CLIP_MM_CACHE = MODELS_DIR / 'multimodal_clip_cache.npz'

if CLIP_MM_CACHE.exists():
    print('Chargement CLIP features depuis le cache...')
    cache = np.load(CLIP_MM_CACHE)
    X_clip_img_train = cache['img_train']
    X_clip_img_val = cache['img_val']
    X_clip_txt_train = cache['txt_train']
    X_clip_txt_val = cache['txt_val']
else:
    print('Chargement du mod\u00e8le CLIP ViT-B/32...')
    clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
        'ViT-B-32', pretrained='laion2b_s34b_b79k'
    )
    clip_model = clip_model.to(DEVICE).eval()
    tokenizer = open_clip.get_tokenizer('ViT-B-32')

    def extract_clip_images(paths, batch_size=64):
        all_feats = []
        for start in range(0, len(paths), batch_size):
            batch = paths[start:start + batch_size]
            imgs = []
            for p in batch:
                try:
                    full = ROOT / p if not Path(p).is_absolute() else Path(p)
                    imgs.append(clip_preprocess(Image.open(full).convert('RGB')))
                except Exception:
                    imgs.append(torch.zeros(3, 224, 224))
            tensor = torch.stack(imgs).to(DEVICE)
            with torch.no_grad():
                feats = clip_model.encode_image(tensor)
            feats = feats / feats.norm(dim=-1, keepdim=True)
            all_feats.append(feats.cpu().numpy())
            if (start // batch_size) % 50 == 0:
                print(f'  Images: {start + len(batch)}/{len(paths)}')
        return np.vstack(all_feats)

    def extract_clip_texts(texts, batch_size=256):
        all_feats = []
        for start in range(0, len(texts), batch_size):
            batch = texts[start:start + batch_size]
            tokens = tokenizer(batch).to(DEVICE)
            with torch.no_grad():
                feats = clip_model.encode_text(tokens)
            feats = feats / feats.norm(dim=-1, keepdim=True)
            all_feats.append(feats.cpu().numpy())
            if (start // batch_size) % 50 == 0:
                print(f'  Textes: {start + len(batch)}/{len(texts)}')
        return np.vstack(all_feats)

    print('Extraction CLIP image features...')
    X_clip_img_train = extract_clip_images(train_df['image_path'].tolist())
    X_clip_img_val = extract_clip_images(val_df['image_path'].tolist())

    print('Extraction CLIP text features...')
    X_clip_txt_train = extract_clip_texts(train_df['text'].tolist())
    X_clip_txt_val = extract_clip_texts(val_df['text'].tolist())

    np.savez(CLIP_MM_CACHE,
             img_train=X_clip_img_train, img_val=X_clip_img_val,
             txt_train=X_clip_txt_train, txt_val=X_clip_txt_val)
    print(f'Cache sauvegard\u00e9 : {CLIP_MM_CACHE}')

print(f'CLIP image features : train={X_clip_img_train.shape}, val={X_clip_img_val.shape}')
print(f'CLIP text features  : train={X_clip_txt_train.shape}, val={X_clip_txt_val.shape}')

## 3. Late Fusion (concaténation de features)

Concaténation des features texte et image, normalisées individuellement (L2), puis entraînement de classifieurs classiques.

| Combinaison | Dimensions |
|---|---|
| TF-IDF (300d) + ResNet50 (2048d) | 2348d |
| TF-IDF (300d) + CLIP image (512d) | 812d |
| TF-IDF (300d) + CLIP image (512d) + CLIP texte (512d) | 1324d |

In [ ]:
fusion_configs = {
    'TF-IDF + ResNet50': (
        np.hstack([X_tfidf_train, X_resnet_train]),
        np.hstack([X_tfidf_val, X_resnet_val])
    ),
    'TF-IDF + CLIP img': (
        np.hstack([X_tfidf_train, X_clip_img_train]),
        np.hstack([X_tfidf_val, X_clip_img_val])
    ),
    'TF-IDF + CLIP img + CLIP txt': (
        np.hstack([X_tfidf_train, X_clip_img_train, X_clip_txt_train]),
        np.hstack([X_tfidf_val, X_clip_img_val, X_clip_txt_val])
    ),
}

classifiers = {
    'Logistic Regression': lambda: LogisticRegression(
        max_iter=1000, C=1.0, class_weight='balanced', random_state=SEED
    ),
    'SVM Linear': lambda: LinearSVC(
        C=1.0, class_weight='balanced', random_state=SEED, dual=False, max_iter=2000
    ),
}

fusion_results = []

for fusion_name, (X_tr, X_va) in fusion_configs.items():
    print(f'\n--- {fusion_name} ({X_tr.shape[1]}d) ---')
    for clf_name, clf_fn in classifiers.items():
        clf = clf_fn()
        clf.fit(X_tr, y_train)
        y_pred = clf.predict(X_va)
        f1 = f1_score(y_val, y_pred, average='macro')
        f1w = f1_score(y_val, y_pred, average='weighted')
        fusion_results.append({
            'Features': fusion_name,
            'Classifier': clf_name,
            'F1_macro': f1,
            'F1_weighted': f1w,
        })
        print(f'  {clf_name:<25} F1-macro={f1:.4f}  F1-weighted={f1w:.4f}')

fusion_df = pd.DataFrame(fusion_results).sort_values('F1_macro', ascending=False)
print(f'\nMeilleure fusion : {fusion_df.iloc[0]["Features"]} + {fusion_df.iloc[0]["Classifier"]} = {fusion_df.iloc[0]["F1_macro"]:.4f}')

## 4. CLIP Multimodal : texte et image dans le même espace

CLIP encode texte et image dans un **espace partagé de 512 dimensions**. On compare deux stratégies de fusion :
- **Concaténation** : `[clip_text ; clip_image]` = 1024d
- **Moyenne** : `(clip_text + clip_image) / 2` = 512d

In [ ]:
clip_fusion_configs = {
    'CLIP concat (1024d)': (
        np.hstack([X_clip_txt_train, X_clip_img_train]),
        np.hstack([X_clip_txt_val, X_clip_img_val])
    ),
    'CLIP moyenne (512d)': (
        (X_clip_txt_train + X_clip_img_train) / 2,
        (X_clip_txt_val + X_clip_img_val) / 2
    ),
}

clip_results = []

# R\u00e9f\u00e9rences unimodales CLIP
for ref_name, X_tr, X_va in [
    ('CLIP image seule', X_clip_img_train, X_clip_img_val),
    ('CLIP texte seul', X_clip_txt_train, X_clip_txt_val),
]:
    for clf_name, clf_fn in classifiers.items():
        clf = clf_fn()
        clf.fit(X_tr, y_train)
        y_pred = clf.predict(X_va)
        f1 = f1_score(y_val, y_pred, average='macro')
        clip_results.append({'Features': ref_name, 'Classifier': clf_name, 'F1_macro': f1})

# Fusions CLIP
for fusion_name, (X_tr, X_va) in clip_fusion_configs.items():
    print(f'\n--- {fusion_name} ---')
    for clf_name, clf_fn in classifiers.items():
        clf = clf_fn()
        clf.fit(X_tr, y_train)
        y_pred = clf.predict(X_va)
        f1 = f1_score(y_val, y_pred, average='macro')
        clip_results.append({'Features': fusion_name, 'Classifier': clf_name, 'F1_macro': f1})
        print(f'  {clf_name:<25} F1-macro={f1:.4f}')

clip_df = pd.DataFrame(clip_results)

# Tableau pivot
pivot = clip_df.pivot_table(index='Features', columns='Classifier', values='F1_macro')
print('\n--- R\u00e9capitulatif CLIP ---')
print(pivot.round(4).to_string())

## 5. Démonstration de matching texte-image

Grâce à l'espace commun de CLIP, on peut mesurer la **similarité cosinus** entre un texte et une image. Cela permet de retrouver les images les plus pertinentes pour une description donnée.

In [ ]:
# --- Top-5 images pour des descriptions textuelles ---
sample_queries = val_df.sample(n=5, random_state=SEED)

fig, axes = plt.subplots(5, 6, figsize=(20, 18))
fig.suptitle('Matching texte -> images (CLIP)', fontsize=16, fontweight='bold')

for row_idx, (_, query) in enumerate(sample_queries.iterrows()):
    query_text = query['text'][:80]
    query_emb = X_clip_txt_val[val_df.index.get_loc(query.name)].reshape(1, -1)
    sims = compute_similarity(query_emb, X_clip_img_val)
    top5_idx = np.argsort(sims)[::-1][:5]

    axes[row_idx][0].text(0.05, 0.5, query_text, fontsize=8, va='center',
                          wrap=True, transform=axes[row_idx][0].transAxes)
    axes[row_idx][0].set_title(f'Classe: {query["prdtypecode"]}', fontsize=8)
    axes[row_idx][0].axis('off')

    for col_idx, img_idx in enumerate(top5_idx):
        ax = axes[row_idx][col_idx + 1]
        img_row = val_df.iloc[img_idx]
        try:
            img_path = ROOT / img_row['image_path'] if not Path(img_row['image_path']).is_absolute() else Path(img_row['image_path'])
            img = Image.open(img_path).convert('RGB').resize((112, 112))
            ax.imshow(img)
        except Exception:
            ax.text(0.5, 0.5, '?', ha='center', va='center')
        match = img_row['prdtypecode'] == query['prdtypecode']
        color = 'green' if match else 'red'
        ax.set_title(f'{sims[img_idx]:.3f} ({img_row["prdtypecode"]})', fontsize=7, color=color)
        ax.axis('off')

plt.tight_layout()
plt.savefig(MODELS_DIR / 'multimodal_matching_demo.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Heatmap de similarit\u00e9 texte-image (\u00e9chantillon) ---
N_SAMPLE = 30
sample_idx = val_df.sample(n=N_SAMPLE, random_state=SEED).index
loc_idx = [val_df.index.get_loc(i) for i in sample_idx]

txt_emb_sample = X_clip_txt_val[loc_idx]
img_emb_sample = X_clip_img_val[loc_idx]
labels_sample = val_df.iloc[loc_idx]['prdtypecode'].values

sim_matrix = txt_emb_sample @ img_emb_sample.T

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(sim_matrix, cmap='RdYlGn', center=0, vmin=-0.1, vmax=0.5,
            xticklabels=labels_sample, yticklabels=labels_sample, ax=ax)
ax.set_xlabel('Images (prdtypecode)')
ax.set_ylabel('Textes (prdtypecode)')
ax.set_title('Similarit\u00e9 cosinus texte-image (CLIP) - diagonale = paires correctes')
plt.tight_layout()
plt.show()

diag = np.diag(sim_matrix)
off_diag = sim_matrix[~np.eye(N_SAMPLE, dtype=bool)]
print(f'Similarit\u00e9 moyenne paires correctes (diagonale) : {diag.mean():.4f}')
print(f'Similarit\u00e9 moyenne paires incorrectes : {off_diag.mean():.4f}')
print(f'Ratio : {diag.mean() / (off_diag.mean() + 1e-8):.2f}x')

In [ ]:
# --- Recall@K sur l'ensemble de validation ---
recall_scores = recall_at_k(
    query_embeddings=X_clip_txt_val,
    gallery_embeddings=X_clip_img_val,
    query_ids=val_df['productid'].values,
    gallery_ids=val_df['productid'].values,
    k_values=[1, 5, 10, 20]
)
print('Recall@K (texte -> image) :')
for k, v in recall_scores.items():
    print(f'  {k} = {v:.4f}')

## 6. Comparaison finale

Tableau récapitulatif de toutes les approches : texte seul, image seule, et fusion multimodale.

In [ ]:
# --- Baselines unimodales ---
unimodal_results = []

# Texte seul (TF-IDF)
for clf_name, clf_fn in classifiers.items():
    clf = clf_fn()
    clf.fit(X_tfidf_train, y_train)
    y_pred = clf.predict(X_tfidf_val)
    f1 = f1_score(y_val, y_pred, average='macro')
    unimodal_results.append({'Approche': 'Texte seul (TF-IDF)', 'Classifier': clf_name, 'F1_macro': f1})

# Image seule (ResNet50)
for clf_name, clf_fn in classifiers.items():
    clf = clf_fn()
    clf.fit(X_resnet_train, y_train)
    y_pred = clf.predict(X_resnet_val)
    f1 = f1_score(y_val, y_pred, average='macro')
    unimodal_results.append({'Approche': 'Image seule (ResNet50)', 'Classifier': clf_name, 'F1_macro': f1})

unimodal_df = pd.DataFrame(unimodal_results)

# --- R\u00e9capitulatif complet ---
all_results = []

# Unimodal
for _, row in unimodal_df.iterrows():
    all_results.append({'Approche': row['Approche'], 'F1_macro': row['F1_macro']})

# CLIP unimodal (meilleur classifieur)
for feat in ['CLIP image seule', 'CLIP texte seul']:
    sub = clip_df[clip_df['Features'] == feat]
    best = sub.sort_values('F1_macro', ascending=False).iloc[0]
    all_results.append({'Approche': feat, 'F1_macro': best['F1_macro']})

# Fusions (meilleur classifieur par config)
for _, row in fusion_df.iterrows():
    all_results.append({'Approche': f"Fusion: {row['Features']} ({row['Classifier']})", 'F1_macro': row['F1_macro']})

# CLIP fusions
for feat in ['CLIP concat (1024d)', 'CLIP moyenne (512d)']:
    sub = clip_df[clip_df['Features'] == feat]
    best = sub.sort_values('F1_macro', ascending=False).iloc[0]
    all_results.append({'Approche': f"Fusion: {feat} ({best['Classifier']})", 'F1_macro': best['F1_macro']})

# D\u00e9dupliquer par approche (garder le meilleur)
final_df = pd.DataFrame(all_results).sort_values('F1_macro', ascending=False).drop_duplicates(subset='Approche')

print('\n' + '=' * 70)
print('  COMPARAISON FINALE : TOUTES APPROCHES')
print('=' * 70)
for _, row in final_df.iterrows():
    bar = '#' * int(row['F1_macro'] * 50)
    print(f"  {row['Approche']:<45} {row['F1_macro']:.4f}  {bar}")

final_df.to_csv(MODELS_DIR / 'multimodal_comparison.csv', index=False)
print(f'\nSauvegard\u00e9 dans models/multimodal_comparison.csv')

In [ ]:
# --- Graphique comparatif ---
plot_df = final_df.sort_values('F1_macro', ascending=True).tail(10)

fig, ax = plt.subplots(figsize=(12, 7))
colors = []
for approche in plot_df['Approche']:
    a = approche.lower()
    if 'fusion' in a:
        colors.append('#2ecc71')
    elif 'image' in a or 'resnet' in a:
        colors.append('#e74c3c')
    elif 'texte' in a or 'tf-idf' in a:
        colors.append('#3498db')
    else:
        colors.append('#9b59b6')

bars = ax.barh(range(len(plot_df)), plot_df['F1_macro'].values, color=colors)
ax.set_yticks(range(len(plot_df)))
ax.set_yticklabels(plot_df['Approche'].values, fontsize=9)
ax.set_xlabel('F1-macro')
ax.set_title('Comparaison des approches : Texte vs Image vs Fusion Multimodale')

for i, (bar, val) in enumerate(zip(bars, plot_df['F1_macro'].values)):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#3498db', label='Texte seul'),
    Patch(facecolor='#e74c3c', label='Image seule'),
    Patch(facecolor='#2ecc71', label='Fusion multimodale'),
    Patch(facecolor='#9b59b6', label='CLIP'),
]
ax.legend(handles=legend_elements, loc='lower right')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(MODELS_DIR / 'multimodal_comparison_chart.png', dpi=150, bbox_inches='tight')
plt.show()

## Conclusions

### Enseignements clés

1. **Le texte domine** : les features textuelles (TF-IDF) sont bien plus discriminantes que les features image pour la classification de produits e-commerce.

2. **La fusion améliore les résultats** : combiner texte et image apporte un gain par rapport au texte seul, confirmant la complémentarité des modalités.

3. **CLIP encodé dans un espace commun** : permet non seulement la classification mais aussi le **matching** (retrouver une image depuis un texte), ouvrant la porte à des applications de recherche produit.

### Limitations

- **45% des produits n'ont pas d'image** : la fusion ne couvre qu'un sous-ensemble des données. En production, un système de fallback texte-seul est nécessaire.
- **CLIP tronque à 77 tokens** : les descriptions longues perdent de l'information.
- **Pas de fine-tuning CLIP** : les performances pourraient être améliorées en adaptant CLIP au domaine e-commerce.